In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import KarateClub
from torch_geometric.nn import GCNConv  # Graph Convolutional Network layer

# ==========================================
# 1. LOAD AND INSPECT THE GRAPH
# ==========================================
dataset = KarateClub()
data = dataset[0]  # The dataset contains exactly 1 graph instance

print("--- Graph Structural Metrics ---")
print(f"Number of nodes (club members): {data.num_nodes}")
print(f"Number of edges (connections) : {data.num_edges}")
print(f"Number of target classes      : {dataset.num_classes} (The 2 factions)")
print(f"Node feature matrix shape     : {data.x.shape}") 
print(f"Edge index matrix shape       : {data.edge_index.shape}\n")

# ==========================================
# 2. DEFINE THE GNN ARCHITECTURE
# ==========================================
class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        torch.manual_seed(1234)
        
        # Define two Graph Convolution layers
        # Layer 1: Inputs node features, outputs a hidden embedding of size 4
        self.conv1 = GCNConv(dataset.num_features, 4)
        # Layer 2: Compresses hidden embedding down to our 2 target classes
        self.conv2 = GCNConv(4, dataset.num_classes)

    def forward(self, x, edge_index):
        # Step 1: Run first message-passing pass & apply non-linear activation
        h = self.conv1(x, edge_index)
        h = torch.tanh(h)
        
        # Step 2: Run second message-passing pass
        h = self.conv2(h, edge_index)
        
        return h

# Instantiate the model and define standard optimization tools
model = GCN()
criterion = torch.nn.CrossEntropyLoss()  # Loss function for classification
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

# ==========================================
# 3. THE TRAINING LOOP
# ==========================================
print("--- Initiating GNN Training Pipeline ---")

# In this classic semi-supervised setup, we only train on 4 known nodes 
# (the leaders of the factions) and let the network propagate to the rest.
for epoch in range(1, 101):
    model.train()
    optimizer.zero_grad()  # Clear cache gradients
    
    # Forward pass: pass node features (x) and edge connections (edge_index)
    out = model(data.x, data.edge_index)
    
    # Calculate loss based only on the subset of nodes we have training labels for
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    
    # Backpropagation
    loss.backward()
    optimizer.step()
    
    # Track evaluation metrics every 20 steps
    if epoch % 20 == 0:
        model.eval()
        pred = out.argmax(dim=1)
        correct = (pred == data.y).sum()
        accuracy = int(correct) / data.num_nodes
        print(f"Epoch: {epoch:3d} | Train Loss: {loss.item():.4f} | Global Node Accuracy: {accuracy * 100:.1f}%")

# ==========================================
# 4. TESTING THE MODEL'S PREDICTIONS
# ==========================================
print("\n--- Model Inference Verification ---")
model.eval()
final_output = model(data.x, data.edge_index)
final_predictions = final_output.argmax(dim=1)

# Print a subset of club members to verify their mapped classifications
for node_id in range(10):
    print(f"Member ID {node_id:2d}: Actual Faction = {data.y[node_id].item()} | Predicted Faction = {final_predictions[node_id].item()}")

--- Graph Structural Metrics ---
Number of nodes (club members): 34
Number of edges (connections) : 156
Number of target classes      : 4 (The 2 factions)
Node feature matrix shape     : torch.Size([34, 34])
Edge index matrix shape       : torch.Size([2, 156])

--- Initiating GNN Training Pipeline ---
Epoch:  20 | Train Loss: 1.2283 | Global Node Accuracy: 50.0%
Epoch:  40 | Train Loss: 0.9621 | Global Node Accuracy: 82.4%
Epoch:  60 | Train Loss: 0.6304 | Global Node Accuracy: 82.4%
Epoch:  80 | Train Loss: 0.3457 | Global Node Accuracy: 82.4%
Epoch: 100 | Train Loss: 0.1833 | Global Node Accuracy: 82.4%

--- Model Inference Verification ---
Member ID  0: Actual Faction = 1 | Predicted Faction = 1
Member ID  1: Actual Faction = 1 | Predicted Faction = 1
Member ID  2: Actual Faction = 1 | Predicted Faction = 0
Member ID  3: Actual Faction = 1 | Predicted Faction = 1
Member ID  4: Actual Faction = 3 | Predicted Faction = 3
Member ID  5: Actual Faction = 3 | Predicted Faction = 3
Member 